# LangChain

https://www.youtube.com/watch?v=hVs8MVydN3A&list=PL4HikwTaYE0GEs7lvlYJQcvKhq0QZGRVn&index=2

LangChain jest biblioteką, która dodaje kolejną warstwę abstrakcji nad API obsługującymi różne modele językowe. Posiada również szereg narzędzi używanych w pracy z LLM takimi jak np. loadery do plików na potrzeby systemu RAG.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

## Quickstart

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model="openai/gpt-4o-mini")
llm

In [ ]:
response = llm.invoke("Hello")
response

In [ ]:
response.content

In [ ]:
response.response_metadata

In [ ]:
response.response_metadata["model_name"]

In [ ]:
response.response_metadata["token_usage"]

---

In [ ]:
response = llm.invoke([
    {"role": "system", "content": "Wyjaśnij zwięźle poniższe pojęcie"},
    {"role": "user", "content": "LangChain"}
])
response

In [ ]:
response.content

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
response = llm.invoke([
    SystemMessage("Wyjaśnij zwięźle poniższe pojęcie"),
    HumanMessage("Langchain"),
])
response

In [ ]:
response.content

## `batch()` i `stream()`

Funkcji `batch()` używamy w celu wygenerowania odpowiedzi na szereg niezależnych wiadomości.

Funkcji `stream()` używamy kiedy chcemy wygenerować odpowiedź w postaci strumienia tokenów.

In [ ]:
import time

In [ ]:
llm = ChatOpenAI(model="openai/gpt-4o-mini")

**`batch()`**

Wszystkie prompty przesłane do `batch()` wykonują się równolegle i stanowią osobne wątki konwersacji.

In [ ]:
start = time.perf_counter()

batch_response = llm.batch(["Wyjaśnij zwięźle czym jest LangChain", "Wyjaśnij zwięźle czym jest RAG", "Opisz metodę invoke() w Langchain"])

stop = time.perf_counter()
print("Czas działania:", round(stop - start, 2))

In [ ]:
for response in batch_response:
    print(response.content, "\n\n-----\n\n")

In [ ]:
start = time.perf_counter()

responses = []
for prompt in ["Wyjaśnij zwięźle czym jest LangChain", "Wyjaśnij zwięźle czym jest RAG", "Opisz metodę invoke() w Langchain"]:
    responses.append(llm.invoke(prompt).content)

stop = time.perf_counter()
print("Czas działania:", round(stop - start, 2))

In [ ]:
responses

---

In [ ]:
batch_response = llm.batch([
    [SystemMessage("Zamień na wielką literę"), HumanMessage("a")],
    [SystemMessage("Zamień na wielką literę"), HumanMessage("b")],
    [SystemMessage("Zamień na wielką literę"), HumanMessage("c")],
])

In [ ]:
for response in batch_response:
    print(response.content, "\n\n-----\n\n")

**`stream()`**

Funkcja `stream()` tworzy generator, po którym możemy iterować. Każdy kolejny token jest generowany przez model dopiero w trakcie iterowania po generatorze.

In [ ]:
llm.stream("Hello")

In [ ]:
for item in llm.stream("Hello"):
    print(item.content, end='')
    time.sleep(0.4)

> **ZADANIA**

## Prompt template i LLM Chain

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

**`PromptTemplate`**

In [ ]:
template = "Wytłumacz następujące pojęcie: {subject}"

In [ ]:
prompt_template = PromptTemplate.from_template(template)

llm = ChatOpenAI(model="openai/gpt-4o-mini")

In [ ]:
chain = prompt_template | llm
type(chain)

In [ ]:
response = chain.invoke({"subject": "Prompt template w Langchain"})

In [ ]:
print(response.content)

**`ChatPromptTemplate`**

In [ ]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [ ]:
messages = [
    SystemMessagePromptTemplate.from_template("Wygeneruj listę {liczba} synonimów oddzielonych znakiem {separator} do podanego słowa"),
    HumanMessagePromptTemplate.from_template("{słowo}")
]

In [ ]:
chat_prompt_template = ChatPromptTemplate.from_messages(messages)
llm = ChatOpenAI(model="openai/gpt-4o-mini")

chain = chat_prompt_template | llm

In [ ]:
response = chain.invoke({"liczba": 3, "separator": ";", "słowo": "LLM"})
response

In [ ]:
response.content

## Structured output w LangChain

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class ResponseFormat(BaseModel):
    word: str = Field(description="Oryginalne słowo")
    synonyms: list[str] = Field(description="Lista synonimów. Każde słowo powinno być otoczone z obu stron przez **")
    antonyms: list[str] = Field(description="Lista antonimów")

In [ ]:
llm = ChatOpenAI(model="openai/gpt-4o-mini").with_structured_output(ResponseFormat)

In [ ]:
response = llm.invoke("Podaj 3 synonimy i 3 antonimy do 'dobry'")
response

In [ ]:
response.model_dump()

> **ZADANIA**